# AXI test 12: does the way a DENSE graph is written change what a stock model can read? (Qwen2.5-7B-Instruct, free Colab T4)
One test, about one idea in a text the owner pasted on 15 Oct 2026: when a graph is **dense** (more than 65 % of all possible edges), writing it as text tuples is wasteful, and the framework should switch to **packed bit matrices (64-bit integers)**. The text calls this "a massive, unmeasured efficiency lever". Nobody has measured it. This notebook measures the part a stock model allows: **can it READ the graph in that form?**

* The SAME graph (24 services, either **dense**: 400 of 552 possible edges, or **sparse**: 138) is written in **five ways**: the repository's text tuples; an adjacency list (each service with who it points to and who points to it); a 0/1 matrix with spaces; the same rows as unbroken 0/1 strings; and each row as one **64-bit integer in hexadecimal** (the format of the text).
* The SAME 8 questions per graph are asked in every format ("which services does X have an edge to?", "which services have an edge to X?", "is there an edge from A to B?"). 5 dense + 3 sparse graphs, 64 questions per format. The model answers freely (no grammar).
* **14 reading rules**, each with my probability, were written and locked BEFORE this run in `docs/DENSE_TEST.md`. Two are plain arithmetic (how many tokens each format needs), the rest are about what the model gets right. I expect the packed forms to save most of the tokens and the model to read them at about chance. A line that comes out NO is a finding, not an error.

**What this does NOT test:** the host side (a bitset is a normal, small structure in Python or Rust and nothing here says otherwise); a model trained to read bitsets; the cut-over at exactly 0.65; writing edits. Only: can a stock model read the packed form?

**What to know**
* **Cost: $0.** A local open model, no key, no paid service. It needs the **free T4 GPU**: Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save.
* **Time (my estimate, not measured):** the long run (Cell 8) 25 to 40 minutes. **Cell 7 measures your machine first and prints the real projection.** If Colab disconnects: reconnect, run Cells 2 to 5 again, then Cell 8 again; each finished graph-and-format is saved to your Google Drive, so it continues where it stopped.
* **Cell 6 is a safety check** (no model time): it stops if the prompts are not what the tokenizer's chat template would build. If it says STOP, send the printout.
* **Do not paste any key or password anywhere.** This notebook needs none.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-tensor.zip` (the new one that comes with this notebook)

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and results are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_dense")

### Cell 5: load the model (downloads about 15 GB once, unless this session already did it for an earlier notebook; about 5 minutes)

In [ ]:
import json, time, torch
from google.colab import files
from axi.experiments.kv_drop import KV_MODEL
from axi.experiments.kv_dense import make_problems, dense_preflight, dense_dry_run, run_unit, run_dense, print_dense, verdict_dense, print_verdict_dense, DENSE_FORMATS, DENSE_MAX_NEW
from axi.experiments.kv_torch import build_kv, make_hf_kit

engine, tok, table, model = build_kv(KV_MODEL, True, "cuda")
kit_plain = make_hf_kit(tok, hint_rules=False)      # only its tokenizer is used here: this test has its own system prompt
dense_problems = make_problems()
T0 = time.time()
def prog(i, n): print(f"  {i}/{n}   {time.time() - T0:6.0f} s", end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
res_dense = None
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1), "| model:", KV_MODEL, "| graphs:", len(dense_problems), "| formats:", len(DENSE_FORMATS))

### Cell 6: SAFETY CHECK (under a minute, no model time). Checks the prompts against the tokenizer's own chat template, that every written graph is read back to exactly its edges, and counts the REAL tokens of the five formats (the pre-registered lines 4 and 5 are about these). If it says STOP, do not run the next cells: send the printout

In [ ]:
checks = dense_preflight(tok, kit_plain, dense_problems)
for name, ok, detail in checks: print(("ok    " if ok else "FAIL  ") + name + "   [" + detail + "]")
json.dump([dict(name=n, ok=bool(o), detail=d) for n, o, d in checks], open("/content/out/axi_dense_preflight.json", "w"), indent=1); store.persist("axi_dense_preflight.json")
assert all(ok for _, ok, _ in checks), "STOP: a check failed. Do not run the next cells. Send the printout above."
real = dense_dry_run(dense_problems, count=lambda t: len(kit_plain.enc(t)))
print("\nREAL tokens of the state (mean over the graphs), by format:")
for reg in ("dense", "sparse"): print(f"  {reg:6s} rho {real[reg]['rho']:.3f}: " + "  ".join(f"{f} {real[reg][f]:.0f} ({real[reg][f + '_over_tuples']:.2f} x tuples)" for f in DENSE_FORMATS))
print("\nALL CHECKS OK. Go on to Cell 7.")

### Cell 7: quick check (about 3 to 6 minutes): one DENSE graph read in two formats, `tuples` and `bitset`, all 8 questions each. It shows what the model answers and tells you how long the long run will take on this machine. If it fails, stop and send me the red error text

In [ ]:
smoke = []
for fmt in ("tuples", "bitset"):
    t0 = time.time(); row = run_unit(engine, kit_plain, table, dense_problems[0], fmt); row["wall"] = time.time() - t0; smoke.append(row)
    print(f"--- graph {row['pid']} written as {fmt}: state {row['state_tokens']} tokens, 8 questions in {row['wall']:.0f} s")
    for q, ref in zip(row["qs"], dense_problems[0]["questions"]):
        print(f"   {q['kind']:5s} truth {str(ref['truth'])[:48]:50s} answer {q['text'].strip()[:60]!r}")
per_unit = sum(r["wall"] for r in smoke) / len(smoke)
print(f"\nOne graph in one format took about {per_unit:.0f} s here (the two formats averaged; the matrix forms are quicker, the tuples slower). The long run has 40 of them: about {40 * per_unit * 0.6 / 60:.0f} to {40 * per_unit / 60:.0f} minutes.")

### Cell 8: THE DENSE-GRAPH TEST. 8 graphs x 5 formats = 40 runs of 8 questions (the counter shows run i/40; each finished run is saved to your Drive)

In [ ]:
T0 = time.time(); name = "axi_dense"
print("progress file:", store.restore(name + ".jsonl"))
res_dense = run_dense(engine, kit_plain, table, store.path(name + ".jsonl"), dense_problems, meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog))
print(); print_dense(res_dense["summary"], "DENSE TEST")
final_dense = verdict_dense(res_dense["summary"]); print_verdict_dense(final_dense, "DENSE TEST")
files.download(save(res_dense, store.path(name + ".json"))); store.persist(name + ".json")
print("peak GPU memory (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 1), "| minutes:", round((time.time() - T0) / 60, 1))
torch.cuda.empty_cache()

### Done
File downloaded: `axi_dense.json` (Cell 8); the checks are also in `axi_dense_preflight.json` on your Drive (folder `axi_dense`). Upload `axi_dense.json` to the chat, with a screenshot of the two printed tables and of the 14 reading-rule lines. Do not paste any keys anywhere.

When the numbers are in, `docs/DENSE_TEST.md` gets a "Result" section **below** the locked predictions; nothing above it changes.